# ANA Educação Financeira — Base Sumarizada por Cliente

Fluxo consolidado para gerar a base oficial `df_base_sumarizada_cliente`, com uma linha por cliente na janela mensal.

A base final reúne:
- resumo técnico das transações;
- perfil de renda e perfil financeiro;
- blocos de entrada e saída por classificação;
- resultado do orçamento e seu status.

Após a consolidação, os próximos indicadores devem usar apenas `vw_base_sumarizada_cliente`, sem retornar à base transacional.

## Escopo e decisões consolidadas

### Estrutura do fluxo

```text
vw_base_transacoes
        ↓
vw_resumo_transacoes_cliente          → resumo técnico por natureza C/D
vw_blocos_classificacao_cliente       → blocos por CD_CLASSIFICACAO_CATEGORIA
        ↓
vw_base_sumarizada_cliente            → base oficial por cliente
        ↓
vw_resultado_orcamento                → resultado derivado da própria sumarizada
        ↓
vw_base_sumarizada_cliente            → base final
```

### Regras mantidas

- O resumo técnico existente continua usando `CD_NTZ_CTB_TRAN` para quantidades e valores gerais de crédito e débito.
- Os blocos classificados usam exclusivamente `CD_CLASSIFICACAO_CATEGORIA`:
  - `0` a `4`: entradas;
  - `5` a `9`: saídas.
- A sumarizada final terá **40 colunas**, em uma única linha por `CD_CLI`.
- O resultado do orçamento usa `VL_ENT_TOTAL` e `VL_SAI_TOTAL`, sem nova leitura da base transacional.
- As faixas de status são parametrizadas, com equilíbrio em torno de `PC_SAI_ENT = 1`.

## 1. Conexão

In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="etl-vinculacao-mf-insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer":
                "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive":"true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%%spark

import os

ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
hoje = ler_variavel_ambiente_spark("HOJE")

if ambiente == "MODELAGEM":
    sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
    database = f"sbx_{sandbox}"
else:
    database = f"hive_{dominio}"

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(env=env_spark)


## 2. Configuração e janela de processamento

In [ ]:
%%spark

from datetime import datetime
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.window import Window

nome_tabela_saida = "ana_edu_fin_cli"
nome_tabela = f"{database}.{nome_tabela_saida}"

fetchsize_db2 = 10_000
num_partitions_db2 = 16
num_partitions_transacoes = 20
partitions_hive = 10
partitions_stage_default = 20

materializar_etapas = True
limpar_cache_intermediario = True
limpar_cache_publicacao = True
storage_level_materializacao = StorageLevel.MEMORY_AND_DISK

diagnostico_opt_in = False

# Faixa padrão para colunas de identificação de cliente.
cd_cli_min = 1
cd_cli_max = 999_999_999

# Faixas do resultado do orçamento:
# - 0,25: distância a partir da qual o superávit/déficit é forte.
# - 0,05: faixa de neutralidade em torno do equilíbrio.
faixa_resultado_forte = 0.25
faixa_resultado_neutro = 0.05

pc_limite_superavit_forte = 1 - faixa_resultado_forte
pc_limite_neutro_inferior = 1 - faixa_resultado_neutro
pc_limite_neutro_superior = 1 + faixa_resultado_neutro
pc_limite_deficit_forte = 1 + faixa_resultado_forte

tabela_spark = nome_tabela

pc_ref_gen = 0.750000
pc_ref_ess = 0.500000
pc_ref_flex = 0.300000
pc_ref_res = 0.200000
pc_ref_cred = 0.300000


In [ ]:
%%spark

data_atual = hoje
data_ini = data_atual[:8] + "01"

df_datas = (
    spark.createDataFrame([(1,)], ["id"])
    .withColumn("data_atual", F.lit(data_atual).cast("date"))
    .withColumn("data_ini", F.lit(data_ini).cast("date"))
    .withColumn("data_fim", F.last_day("data_atual"))
    .select("data_atual", "data_ini", "data_fim")
)

linha_datas = df_datas.collect()[0]

data_ini = datetime.strftime(linha_datas[1], "%Y-%m-%d")
data_fim = datetime.strftime(linha_datas[2], "%Y-%m-%d")
ano_etl = int(datetime.strftime(linha_datas[1], "%Y"))

print(f"Data de Inicio da query: {data_ini}")
print(f"Data final da query: {data_fim}")
print(f"Ano do ETL: {ano_etl}")


## 3. Base transacional

In [ ]:
%%spark

query_base_transacoes = f"""
SELECT
    a.CD_CLI,

    a.CD_PRD,

    CASE
        WHEN a.CD_PRD = 6 THEN 'CONTA CORRENTE'
        WHEN a.CD_PRD = 9 THEN 'CARTAO'
        ELSE NULL
    END AS NM_PRD,

    a.VL_TRAN,

    a.CD_TIP_MOE_CRR,

    CASE
        WHEN a.CD_TIP_MOE_CRR IS NULL THEN CAST(NULL AS VARCHAR(100))
        ELSE CAST(NULL AS VARCHAR(100))
    END AS NM_TIP_MOE_CRR,

    a.CD_NTZ_CTB_TRAN,

    CASE
        WHEN a.CD_NTZ_CTB_TRAN = 'C' THEN 'CREDITO'
        WHEN a.CD_NTZ_CTB_TRAN = 'D' THEN 'DEBITO'
        ELSE NULL
    END AS NM_TP_LANCAMENTO,

    a.CD_CTGR_TRAN_OGNL

FROM DB2GFP.TRAN_RLZD_INST_PCT a

WHERE a.DT_TRAN BETWEEN '{data_ini}' AND '{data_fim}'
"""

df_base_transacoes = cliente_db2.run_select(
    query_base_transacoes,
    fetchsize=fetchsize_db2,
    partition_column="CD_CLI",
    lower_bound=17,
    upper_bound=932_774_311,
    num_partitions=num_partitions_transacoes,
)


## 4. Classificação transacional por categoria e natureza

In [ ]:
%%spark

# ============================================================
# 1. Dicionários de códigos por tipo de classificação
# ============================================================

codigos_classificacao_credito = {
    "Transferência / entrada indefinida": 0,
    "Receita / rendimento / benefício": 1,
    "Restituição / estorno / reembolso / ajuste": 2,
    "Resgate de investimento": 3,
    "Crédito tomado / liberação de crédito": 4,
}

codigos_classificacao_debito = {
    "Neutro / não classificado": 5,
    "Essenciais": 6,
    "Flexíveis": 7,
    "Futuro": 8,
    "Dívidas / crédito / custo financeiro": 9,
}


# ============================================================
# 2. Mapa de categorias transacionais
#
# Estrutura:
# (
#     CD_GRUPO_CATEGORIA,
#     CD_CTGR_TRAN_OGNL,
#     NM_CATEGORIA,
#     NM_CLASSIFICACAO_CREDITO,
#     NM_CLASSIFICACAO_DEBITO
# )
# ============================================================

base_variaveis_classificacao = [
    (0, 0, "Sem categoria", "Transferência / entrada indefinida", "Neutro / não classificado"),
    (0, 83, "Sem categoria", "Transferência / entrada indefinida", "Neutro / não classificado"),

    (1, 1, "Salário", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (1, 2, "Vale Alimentação", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (1, 3, "Restituição de IR", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),
    (1, 4, "Bonificação", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (1, 5, "Outros Rendimentos", "Receita / rendimento / benefício", "Neutro / não classificado"),

    (2, 6, "Água", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (2, 7, "Eletricidade e Gás", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (2, 9, "Compra de Imóvel", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro"),
    (2, 10, "Aluguel e Condomínio", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (2, 11, "Móveis e Utensílios", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (2, 12, "Serviços e Manutenção", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (2, 13, "Empregados", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (2, 14, "Animais e Pets", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (2, 3790, "Seguro Residencial", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),

    (3, 15, "Educação Superior", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (3, 16, "Colégio", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (3, 17, "Idiomas", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (3, 18, "Publicações e Papelaria", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (3, 20, "Outros Gastos", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),

    (4, 21, "Viagens e Lazer", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (4, 22, "Esportes e Academia", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (4, 25, "Cultura e Entretenimento", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (4, 26, "Publicações Digitais", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (4, 61, "Jogos e Loterias", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),

    (5, 27, "Plano de Saúde", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (5, 28, "Serviços de Saúde", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (5, 29, "Dentista", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (5, 30, "Farmácias e Drogarias", "Restituição / estorno / reembolso / ajuste", "Essenciais"),

    (6, 32, "Feira e Supermercado", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (6, 35, "Bar", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),

    (7, 36, "Compra de Veículo", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro"),
    (7, 37, "Combustível", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (7, 38, "Estacionamento e Pedágio", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (7, 39, "Seguro de Veículo", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (7, 40, "Serviços e Manutenção", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (7, 41, "Transporte Urbano e Apps", "Restituição / estorno / reembolso / ajuste", "Essenciais"),

    (8, 42, "Vestuário e Acessórios", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 43, "Cuidado Pessoal e Beleza", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 44, "Compras Diversas", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),
    (8, 45, "Pensão Alimentícia", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (8, 46, "Seguros e Previdência", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),
    (8, 47, "Doação", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 48, "Gasto com Familiares", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 49, "Presentes", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 60, "Serviços diversos", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),
    (8, 4417, "Empréstimos e Prestações", "Crédito tomado / liberação de crédito", "Dívidas / crédito / custo financeiro"),

    (9, 51, "Telefonia e Internet", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (9, 53, "Assinatura TV e Streaming", "Restituição / estorno / reembolso / ajuste", "Flexíveis"),

    (10, 54, "IPTU", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (10, 55, "IPVA e Gastos Detran", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (10, 56, "Imposto de Renda", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (10, 57, "ISS(Imposto sobre Serviços)", "Restituição / estorno / reembolso / ajuste", "Essenciais"),
    (10, 58, "GPS(Guia de Previdência Social)", "Restituição / estorno / reembolso / ajuste", "Futuro"),
    (10, 59, "Serviços Financeiros", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro"),
    (10, 3787, "IOF", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro"),
    (10, 3788, "Encargos e Tarifas", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro"),

    (11, 279, "Gastos Diversos", "Transferência / entrada indefinida", "Neutro / não classificado"),
    (11, 39434, "Cheque", "Transferência / entrada indefinida", "Neutro / não classificado"),
    (11, 39435, "Saque", "Transferência / entrada indefinida", "Neutro / não classificado"),
    (11, 39436, "Transferência", "Transferência / entrada indefinida", "Neutro / não classificado"),
    (11, 39437, "Boletos Diversos", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),

    (12, 111, "Cartão de Crédito", "Restituição / estorno / reembolso / ajuste", "Dívidas / crédito / custo financeiro"),

    (13, 448977, "Aplicação", "Restituição / estorno / reembolso / ajuste", "Futuro"),
    (13, 448978, "Resgate de Investimentos", "Resgate de investimento", "Neutro / não classificado"),

    (14, 300, "Receitas Agro", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (14, 310, "Criações", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (14, 330, "Cultivos", "Receita / rendimento / benefício", "Neutro / não classificado"),
    (14, 350, "Insumos", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),
    (14, 370, "Apoio Produtivo", "Restituição / estorno / reembolso / ajuste", "Neutro / não classificado"),
]


# ============================================================
# 3. Validações da estrutura Python
# ============================================================

codigos_categoria = [
    int(linha[1])
    for linha in base_variaveis_classificacao
]

if len(codigos_categoria) != len(set(codigos_categoria)):
    raise ValueError(
        "Existem códigos duplicados em CD_CTGR_TRAN_OGNL "
        "na base de classificação."
    )

classificacoes_credito_utilizadas = {
    linha[3]
    for linha in base_variaveis_classificacao
}

classificacoes_debito_utilizadas = {
    linha[4]
    for linha in base_variaveis_classificacao
}

classificacoes_credito_ausentes = (
    classificacoes_credito_utilizadas
    - set(codigos_classificacao_credito)
)

classificacoes_debito_ausentes = (
    classificacoes_debito_utilizadas
    - set(codigos_classificacao_debito)
)

if classificacoes_credito_ausentes:
    raise ValueError(
        "Classificações de crédito sem código definido: "
        f"{sorted(classificacoes_credito_ausentes)}"
    )

if classificacoes_debito_ausentes:
    raise ValueError(
        "Classificações de débito sem código definido: "
        f"{sorted(classificacoes_debito_ausentes)}"
    )


# ============================================================
# 4. Dicionário final de classificação
#
# Chave:
# (CD_CTGR_TRAN_OGNL, CD_NTZ_CTB_TRAN)
#
# Valor:
# (
#     CD_CLASSIFICACAO_CATEGORIA,
#     NM_CLASSIFICACAO_CATEGORIA
# )
# ============================================================

mapa_classificacao_categoria = {}

for (
    _,
    cd_ctgr_tran_ognl,
    _,
    nm_classificacao_credito,
    nm_classificacao_debito,
) in base_variaveis_classificacao:

    cd_ctgr_tran_ognl = int(cd_ctgr_tran_ognl)

    mapa_classificacao_categoria[
        (cd_ctgr_tran_ognl, "C")
    ] = (
        int(
            codigos_classificacao_credito[
                nm_classificacao_credito
            ]
        ),
        nm_classificacao_credito,
    )

    mapa_classificacao_categoria[
        (cd_ctgr_tran_ognl, "D")
    ] = (
        int(
            codigos_classificacao_debito[
                nm_classificacao_debito
            ]
        ),
        nm_classificacao_debito,
    )


# ============================================================
# 5. Conversão do dicionário para DataFrame de apoio
# ============================================================

linhas_mapa_classificacao = [
    (
        cd_ctgr_tran_ognl,
        cd_ntz_ctb_tran,
        cd_classificacao_categoria,
        nm_classificacao_categoria,
    )
    for (
        cd_ctgr_tran_ognl,
        cd_ntz_ctb_tran,
    ), (
        cd_classificacao_categoria,
        nm_classificacao_categoria,
    ) in sorted(mapa_classificacao_categoria.items())
]

df_mapa_classificacao_categoria = spark.createDataFrame(
    linhas_mapa_classificacao,
    """
        CD_CTGR_TRAN_OGNL BIGINT,
        CD_NTZ_CTB_TRAN STRING,
        CD_CLASSIFICACAO_CATEGORIA BIGINT,
        NM_CLASSIFICACAO_CATEGORIA STRING
    """,
)

df_mapa_classificacao_categoria.createOrReplaceTempView(
    "vw_mapa_classificacao_categoria"
)


# ============================================================
# 6. Aplicação da classificação na base transacional
# ============================================================

codigo_fallback_credito = (
    codigos_classificacao_credito[
        "Transferência / entrada indefinida"
    ]
)

codigo_fallback_debito = (
    codigos_classificacao_debito[
        "Neutro / não classificado"
    ]
)

nome_fallback_credito = (
    "Transferência / entrada indefinida"
)

nome_fallback_debito = (
    "Neutro / não classificado"
)

df_base_transacoes_classificada = (
    df_base_transacoes.alias("b")
    .join(
        F.broadcast(
            df_mapa_classificacao_categoria
        ).alias("m"),
        (
            F.col("b.CD_CTGR_TRAN_OGNL").cast("bigint")
            == F.col("m.CD_CTGR_TRAN_OGNL")
        )
        & (
            F.col("b.CD_NTZ_CTB_TRAN")
            == F.col("m.CD_NTZ_CTB_TRAN")
        ),
        "left",
    )
    .select(
        F.col("b.CD_CLI"),
        F.col("b.CD_PRD"),
        F.col("b.NM_PRD"),
        F.col("b.VL_TRAN"),
        F.col("b.CD_TIP_MOE_CRR"),
        F.col("b.NM_TIP_MOE_CRR"),
        F.col("b.CD_NTZ_CTB_TRAN"),
        F.col("b.NM_TP_LANCAMENTO"),
        F.col("b.CD_CTGR_TRAN_OGNL"),

        F.coalesce(
            F.col("m.CD_CLASSIFICACAO_CATEGORIA"),

            F.when(
                F.col("b.CD_NTZ_CTB_TRAN") == "C",
                F.lit(codigo_fallback_credito).cast("bigint"),
            )
            .when(
                F.col("b.CD_NTZ_CTB_TRAN") == "D",
                F.lit(codigo_fallback_debito).cast("bigint"),
            )
            .otherwise(
                F.lit(None).cast("bigint")
            ),
        ).alias("CD_CLASSIFICACAO_CATEGORIA"),

        F.coalesce(
            F.col("m.NM_CLASSIFICACAO_CATEGORIA"),

            F.when(
                F.col("b.CD_NTZ_CTB_TRAN") == "C",
                F.lit(nome_fallback_credito),
            )
            .when(
                F.col("b.CD_NTZ_CTB_TRAN") == "D",
                F.lit(nome_fallback_debito),
            )
            .otherwise(
                F.lit(None).cast("string")
            ),
        ).alias("NM_CLASSIFICACAO_CATEGORIA"),
    )
)


# ============================================================
# 7. Substitui a base e recria a view usada nas próximas etapas
# ============================================================

df_base_transacoes = df_base_transacoes_classificada

df_base_transacoes.createOrReplaceTempView(
    "vw_base_transacoes"
)


## 5. Resumo técnico das transações por cliente

In [ ]:
%%spark

df_resumo_transacoes_cliente = spark.sql("""
    SELECT
        CD_CLI,

        -- Quantidade total
        COUNT(*) AS QT_TRANS_TOTAL,

        -- Quantidades de entrada
        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1
            END
        ) AS QT_TRANS_ENT,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C'
                 AND CD_PRD = 6
                THEN 1
            END
        ) AS QTD_TRANS_ENT_CC,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'C'
                 AND CD_PRD = 9
                THEN 1
            END
        ) AS QTD_TRANS_ENT_CD,

        -- Quantidades de saída
        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1
            END
        ) AS QT_TRANS_SAI,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D'
                 AND CD_PRD = 6
                THEN 1
            END
        ) AS QTD_TRANS_SAI_CC,

        COUNT(
            CASE
                WHEN CD_NTZ_CTB_TRAN = 'D'
                 AND CD_PRD = 9
                THEN 1
            END
        ) AS QTD_TRANS_SAI_CD,

        -- Valores de entrada
        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                     AND CD_PRD = 6
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT_CC,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'C'
                     AND CD_PRD = 9
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_ENT_CD,

        -- Valores de saída
        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                     AND CD_PRD = 6
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI_CC,

        CAST(
            SUM(
                CASE
                    WHEN CD_NTZ_CTB_TRAN = 'D'
                     AND CD_PRD = 9
                    THEN VL_TRAN
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(25,2)
        ) AS VL_TRANS_SAI_CD

    FROM vw_base_transacoes

    GROUP BY
        CD_CLI
""")

df_resumo_transacoes_cliente.createOrReplaceTempView(
    "vw_resumo_transacoes_cliente"
)


## 6. Perfil de renda e gradeira financeira

In [ ]:
%%spark

query_perfil_renda = """
SELECT
    perfil_mais_recente.CD_CLI,
    perfil_mais_recente.CD_SGM_CLI AS CD_PRFL

FROM (
    SELECT
        cs.CD_CLI,
        cs.CD_SGM_CLI,

        ROW_NUMBER() OVER (
            PARTITION BY cs.CD_CLI
            ORDER BY cs.DT_ENQ_CLI_SGM DESC
        ) AS NR_ORDEM

    FROM DB2SMI.CLI_SGM cs

    WHERE cs.CD_CRIT_SGM_CLI = 2
      AND cs.DT_ENQ_CLI_SGM > DATE('2025-01-01')
) perfil_mais_recente

WHERE perfil_mais_recente.NR_ORDEM = 1
"""

df_perfil_renda = cliente_db2.run_select(
    query_perfil_renda,
    fetchsize=fetchsize_db2,
)

df_perfil_renda.createOrReplaceTempView(
    "vw_perfil_renda"
)


In [ ]:
%%spark

df_gradeira_financeira = spark.sql("""
    SELECT
        CAST(CD_CLI AS BIGINT) AS CD_CLI,
        CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
        CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
        CAST(DT_REF AS DATE) AS DT_REF_GRADEIRA

    FROM (
        SELECT
            CD_CLI,
            CD_MAC_PRFL_CLI,
            CD_MIC_PRFL_CLI,
            DT_REF,

            ROW_NUMBER() OVER (
                PARTITION BY CD_CLI
                ORDER BY
                    CAST(DT_REF AS DATE) DESC NULLS LAST,
                    CD_MAC_PRFL_CLI DESC,
                    CD_MIC_PRFL_CLI DESC
            ) AS NR_ORDEM

        FROM sbx_t2i2016.DVS_GRDR_FNCO_PF
    ) gradeira_mais_recente

    WHERE NR_ORDEM = 1
""")

df_gradeira_financeira.createOrReplaceTempView(
    "vw_gradeira_financeira"
)


## 7. Base sumarizada oficial por cliente

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    WITH perfil_renda AS (
        SELECT
            CAST(CD_CLI AS BIGINT) AS CD_CLI,
            CAST(CD_PRFL AS INT) AS CD_PRFL,

            CASE
                WHEN CD_PRFL = 2502 THEN 'SEM PERFIL'
                WHEN CD_PRFL = 2012 THEN 'PF A'
                WHEN CD_PRFL = 2112 THEN 'PF B'
                WHEN CD_PRFL = 2602 THEN 'PF C'
                WHEN CD_PRFL = 2702 THEN 'PF D'
                WHEN CD_PRFL = 2712 THEN 'PF E'
                WHEN CD_PRFL = 1111 THEN 'A CLASSIFICAR'
                ELSE 'A CLASSIFICAR'
            END AS NM_PRFL

        FROM vw_perfil_renda
    ),

    gradeira_financeira AS (
        SELECT
            CD_CLI,
            CD_MAC_PRFL_CLI,

            CASE
                WHEN CD_MAC_PRFL_CLI = 1 THEN 'Endividado'
                WHEN CD_MAC_PRFL_CLI = 2 THEN 'Equilibrista'
                WHEN CD_MAC_PRFL_CLI = 3 THEN 'Investidor'
                ELSE 'A CLASSIFICAR'
            END AS NM_MAC_PRFL_CLI,

            CD_MIC_PRFL_CLI,

            CASE
                WHEN CD_MIC_PRFL_CLI = 1 THEN 'Inadimplente'
                WHEN CD_MIC_PRFL_CLI = 2 THEN 'Acrobata'
                WHEN CD_MIC_PRFL_CLI = 3 THEN 'Iminente'
                WHEN CD_MIC_PRFL_CLI = 4 THEN 'Consciente'
                WHEN CD_MIC_PRFL_CLI = 5 THEN 'Equilibrista'
                WHEN CD_MIC_PRFL_CLI = 6 THEN 'Acelerado'
                WHEN CD_MIC_PRFL_CLI = 7 THEN 'Precavido'
                WHEN CD_MIC_PRFL_CLI = 8 THEN 'Despreocupado'
                ELSE 'A CLASSIFICAR'
            END AS NM_MIC_PRFL_CLI

        FROM vw_gradeira_financeira
    )

    SELECT
        b.*,

        pr.CD_PRFL,
        COALESCE(pr.NM_PRFL, 'A CLASSIFICAR') AS NM_PRFL,

        gf.CD_MAC_PRFL_CLI,
        COALESCE(
            gf.NM_MAC_PRFL_CLI,
            'A CLASSIFICAR'
        ) AS NM_MAC_PRFL_CLI,

        gf.CD_MIC_PRFL_CLI,
        COALESCE(
            gf.NM_MIC_PRFL_CLI,
            'A CLASSIFICAR'
        ) AS NM_MIC_PRFL_CLI,

        -- Convenção do código unificado: macro * 10 + micro.
        CASE
            WHEN COALESCE(
                gf.NM_MAC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            OR COALESCE(
                gf.NM_MIC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            THEN CAST(NULL AS BIGINT)

            ELSE CAST(
                (gf.CD_MAC_PRFL_CLI * 10)
                + gf.CD_MIC_PRFL_CLI
                AS BIGINT
            )
        END AS CD_PRFL_FIN,

        CASE
            WHEN COALESCE(
                gf.NM_MAC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            OR COALESCE(
                gf.NM_MIC_PRFL_CLI,
                'A CLASSIFICAR'
            ) = 'A CLASSIFICAR'
            THEN 'A CLASSIFICAR'

            ELSE CONCAT(
                gf.NM_MAC_PRFL_CLI,
                ' ',
                gf.NM_MIC_PRFL_CLI
            )
        END AS NM_PRFL_FIN

    FROM vw_resumo_transacoes_cliente b

    LEFT JOIN perfil_renda pr
        ON CAST(b.CD_CLI AS BIGINT) = pr.CD_CLI

    LEFT JOIN gradeira_financeira gf
        ON CAST(b.CD_CLI AS BIGINT) = gf.CD_CLI
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 8. Blocos classificados por cliente

Esta etapa é separada do resumo técnico. Ela sumariza as transações por `CD_CLASSIFICACAO_CATEGORIA` e produz uma linha por cliente para posterior incorporação na base oficial.

In [ ]:
%%spark

df_blocos_classificacao_cliente = spark.sql("""
    SELECT
        CD_CLI,

        -- Blocos de entrada
        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 1
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_REC,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 2
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_REEMB,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 3
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_RESG,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 0
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_IND,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 4
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_ENT_EMPR,

        -- Blocos de saída
        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 5
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_GEN,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 6
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_ESS,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 7
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_FLEX,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 8
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_RES,

        CAST(
            SUM(
                CASE
                    WHEN CD_CLASSIFICACAO_CATEGORIA = 9
                    THEN COALESCE(VL_TRAN, CAST(0 AS DECIMAL(15,2)))
                    ELSE CAST(0 AS DECIMAL(15,2))
                END
            ) AS DECIMAL(18,2)
        ) AS VL_SAI_DIV

    FROM vw_base_transacoes

    GROUP BY
        CD_CLI
""")

df_blocos_classificacao_cliente.createOrReplaceTempView(
    "vw_blocos_classificacao_cliente"
)


### Incorporação dos blocos na sumarizada

Os totais `VL_ENT_TOTAL` e `VL_SAI_TOTAL` são calculados a partir dos blocos classificados. Essa é a última etapa que utiliza a base de transações antes dos indicadores derivados.

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        COALESCE(
            c.VL_ENT_REC,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_REC,

        COALESCE(
            c.VL_ENT_REEMB,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_REEMB,

        COALESCE(
            c.VL_ENT_RESG,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_RESG,

        COALESCE(
            c.VL_ENT_IND,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_IND,

        COALESCE(
            c.VL_ENT_EMPR,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_ENT_EMPR,

        CAST(
            COALESCE(c.VL_ENT_REC, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_REEMB, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_RESG, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_IND, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_ENT_EMPR, CAST(0 AS DECIMAL(18,2)))
            AS DECIMAL(18,2)
        ) AS VL_ENT_TOTAL,

        COALESCE(
            c.VL_SAI_GEN,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_GEN,

        COALESCE(
            c.VL_SAI_ESS,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_ESS,

        COALESCE(
            c.VL_SAI_FLEX,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_FLEX,

        COALESCE(
            c.VL_SAI_RES,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_RES,

        COALESCE(
            c.VL_SAI_DIV,
            CAST(0 AS DECIMAL(18,2))
        ) AS VL_SAI_DIV,

        CAST(
            COALESCE(c.VL_SAI_GEN, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_ESS, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_FLEX, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_RES, CAST(0 AS DECIMAL(18,2)))
            + COALESCE(c.VL_SAI_DIV, CAST(0 AS DECIMAL(18,2)))
            AS DECIMAL(18,2)
        ) AS VL_SAI_TOTAL

    FROM vw_base_sumarizada_cliente b

    LEFT JOIN vw_blocos_classificacao_cliente c
        ON CAST(b.CD_CLI AS BIGINT) = CAST(c.CD_CLI AS BIGINT)
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)


## 9. Resultado do orçamento

O resultado é calculado exclusivamente a partir da sumarizada:

```text
VL_RES_ORC = VL_ENT_TOTAL - VL_SAI_TOTAL
PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL
```

### Faixas de classificação

| Intervalo de `PC_SAI_ENT` | Resultado final |
|---:|---|
| `< 0,75` | Superavitário Forte |
| `0,75` até `< 0,95` | Superavitário Fraco |
| `0,95` até `< 1,00` | Neutro Fraco |
| `1,00` até `1,05` | Neutro Forte |
| `> 1,05` até `1,25` | Deficitário Fraco |
| `> 1,25` | Deficitário Forte |

Para os casos sem entrada, a divisão não é calculada:
- entrada e saída iguais a zero: `Neutro Fraco`;
- entrada igual a zero e saída positiva: `Deficitário Forte`.


In [ ]:
%%spark

df_resultado_orcamento = spark.sql(f"""
    WITH resultado_base AS (
        SELECT
            CD_CLI,

            CAST(
                VL_ENT_TOTAL - VL_SAI_TOTAL
                AS DECIMAL(18,2)
            ) AS VL_RES_ORC,

            CAST(
                CASE
                    WHEN VL_ENT_TOTAL <> 0
                    THEN VL_SAI_TOTAL / VL_ENT_TOTAL
                    ELSE NULL
                END
                AS DECIMAL(9,6)
            ) AS PC_SAI_ENT,

            VL_ENT_TOTAL,
            VL_SAI_TOTAL

        FROM vw_base_sumarizada_cliente
    ),

    resultado_classificado AS (
        SELECT
            CD_CLI,
            VL_RES_ORC,
            PC_SAI_ENT,

            CASE
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 0

                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 2

                WHEN PC_SAI_ENT < {pc_limite_neutro_inferior}
                THEN 1

                WHEN PC_SAI_ENT > {pc_limite_neutro_superior}
                THEN 2

                ELSE 0
            END AS CD_RES_ORC,

            CASE
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 'Neutro'

                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 'Deficitário'

                WHEN PC_SAI_ENT < {pc_limite_neutro_inferior}
                THEN 'Superavitário'

                WHEN PC_SAI_ENT > {pc_limite_neutro_superior}
                THEN 'Deficitário'

                ELSE 'Neutro'
            END AS TX_RES_ORC,

            CASE
                -- Sem entradas e sem saídas
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL = 0
                THEN 'Fraco'

                -- Sem entrada e com saída
                WHEN VL_ENT_TOTAL = 0
                 AND VL_SAI_TOTAL > 0
                THEN 'Forte'

                -- Superavitário forte
                WHEN PC_SAI_ENT < {pc_limite_superavit_forte}
                THEN 'Forte'

                -- Superavitário fraco ou neutro fraco
                WHEN PC_SAI_ENT < 1
                THEN 'Fraco'

                -- Neutro forte
                WHEN PC_SAI_ENT <= {pc_limite_neutro_superior}
                THEN 'Forte'

                -- Deficitário fraco
                WHEN PC_SAI_ENT <= {pc_limite_deficit_forte}
                THEN 'Fraco'

                -- Deficitário forte
                ELSE 'Forte'
            END AS TX_STS_RES

        FROM resultado_base
    )

    SELECT
        CD_CLI,
        VL_RES_ORC,
        CAST(CD_RES_ORC AS INT) AS CD_RES_ORC,
        TX_RES_ORC,
        PC_SAI_ENT,
        TX_STS_RES,

        CONCAT(
            TX_RES_ORC,
            ' ',
            TX_STS_RES
        ) AS TX_STS_FINAL

    FROM resultado_classificado
""")

df_resultado_orcamento.createOrReplaceTempView(
    "vw_resultado_orcamento"
)


### Incorporação do resultado na sumarizada

In [ ]:
%%spark

df_base_sumarizada_cliente = spark.sql("""
    SELECT
        b.*,

        r.VL_RES_ORC,
        r.CD_RES_ORC,
        r.TX_RES_ORC,
        r.PC_SAI_ENT,
        r.TX_STS_RES,
        r.TX_STS_FINAL

    FROM vw_base_sumarizada_cliente b

    INNER JOIN vw_resultado_orcamento r
        ON CAST(b.CD_CLI AS BIGINT) = CAST(r.CD_CLI AS BIGINT)
""")

df_base_sumarizada_cliente.createOrReplaceTempView(
    "vw_base_sumarizada_cliente"
)
